# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

Do **not** Run all. One session = setup cells + **one** sweep cell + the save at the bottom of that cell.

Free Colab dies after 1–2 hours **or** when GPU quota is exhausted. Each item is written to Drive as `{run_id}.partial.jsonl`. If the runtime dies at `[520/800]`, the next session resumes at 521 — **after** `git pull` so that checkpoint code is installed.

Do not treat notebook stdout as a result. Only `results/runs/*.json` plus a registry row is a finished cell.

Next session: setup (Drive restore), then the **same** sweep cell if a `.partial.jsonl` exists, else the next unfinished cell. Finished `config_hash` rows skip.

Use `results/registry_paper.jsonl` (not the n=4 smoke `registry.jsonl`).

In [1]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"
!git log -1 --oneline

Cloning into 'apertus-eval-prep'...
remote: Enumerating objects: 287, done.
remote: Counting objects: 100% (287/287), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 287 (delta 127), reused 239 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (287/287), 620.07 KiB | 15.12 MiB/s, done.
Resolving deltas: 100% (127/127), done.
/content/apertus-eval-prep
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━

In [2]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

Tesla T4
official slices already on disk


## Persist to Google Drive

Authorize Drive when prompted. Files live in `MyDrive/apertus-eval-prep-paper/` so a runtime reset does not wipe finished cells.

Also download the zip to your Mac as a second copy.

In [3]:
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run the clone/pip cell first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    """Run in-process so Colab shows [1/800] live. subprocess.check_call hid all logs."""
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --out-dir results/runs --registry results/registry_paper.jsonl | head -n 50

cwd: /content/apertus-eval-prep
Mounted at /content/drive
partial checkpoints on disk: 0
skip SmolLM2-1.7B-Instruct_control_control_24ffe98d9250761d factor=control=control
skip Qwen2.5-3B-Instruct_control_control_cff017903a47abb9 factor=control=control
run  Phi-3.5-mini-instruct_control_control_31791224954ba45c factor=control=control
run  SmolLM2-1.7B-Instruct_prompt_id_concise_a0852ca6fc3e5c08 factor=prompt_id=concise
run  Qwen2.5-3B-Instruct_prompt_id_concise_e4980863069a102c factor=prompt_id=concise
run  Phi-3.5-mini-instruct_prompt_id_concise_37963fc49f7e8eca factor=prompt_id=concise
run  SmolLM2-1.7B-Instruct_prompt_id_5shot_b6968af4b73708f7 factor=prompt_id=5shot
run  Qwen2.5-3B-Instruct_prompt_id_5shot_8b703d7cb8d9627a factor=prompt_id=5shot
run  Phi-3.5-mini-instruct_prompt_id_5shot_e8c0aa94458abbd2 factor=prompt_id=5shot
run  SmolLM2-1.7B-Instruct_seed_1_4ae7b5af354b3423 factor=seed=1
run  Qwen2.5-3B-Instruct_seed_1_e8b596c4b1a8a527 factor=seed=1
run  Phi-3.5-mini-instruct_see

## Session cells (run one per Colab window)

**Already on GitHub:** SmolLM2-1.7B **control** (318/800) and Qwen2.5-3B **control** (515/800). Those cells should `skip`.

| When | Cell |
|---|---|
| Done | SmolLM2 control, Qwen 3B control |
| **Today / next session** | **Phi-3.5 control** |
| Later | remaining factors / 7B int4 |

Wait until `[800/800]` and the Drive zip download finish before closing the tab. If Colab says GPU time is up, stop — do not start another GPU session the same day. Next GPU window: git pull, Drive cell, **same Phi cell** (resumes from `.partial.jsonl` if Drive has it).

In [ ]:
# DONE on GitHub — skip. Only run if dry-run says run not skip.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — Qwen 3B control (515/800). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "control")

In [4]:
# TODAY — Phi-3.5 control (native transformers; do not use Hub modeling_phi3.py)
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "control")

cwd: /content/apertus-eval-prep
sweep in-process model=microsoft/Phi-3.5-mini-instruct factor=control
run  Phi-3.5-mini-instruct_control_control_31791224954ba45c factor=control=control


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

[1/800] arc_easy/eval/Mercury_SC_409038 correct=True pred='A' ttft_ms=1639.12
[2/800] arc_easy/eval/MCAS_2001_5_19 correct=True pred='C' ttft_ms=215.5
[3/800] arc_easy/eval/NYSEDREGENTS_2008_4_21 correct=True pred='C' ttft_ms=367.92
[4/800] arc_easy/eval/Mercury_7115098 correct=True pred='A' ttft_ms=365.8
[5/800] arc_easy/eval/MCAS_2000_8_34 correct=True pred='C' ttft_ms=371.69
[6/800] arc_easy/eval/Mercury_SC_405883 correct=True pred='D' ttft_ms=376.36
[7/800] arc_easy/eval/Mercury_401762 correct=False pred='A' ttft_ms=382.6
[8/800] arc_easy/eval/Mercury_SC_410624 correct=True pred='A' ttft_ms=399.26
[9/800] arc_easy/eval/Mercury_SC_406688 correct=True pred='D' ttft_ms=402.33
[10/800] arc_easy/eval/Mercury_400556 correct=True pred='D' ttft_ms=405.46
[11/800] arc_easy/eval/Mercury_7219713 correct=True pred='D' ttft_ms=423.15
[12/800] arc_easy/eval/ACTAAP_2014_5_13 correct=True pred='A' ttft_ms=415.02
[13/800] arc_easy/eval/MCAS_2004_5_14 correct=True pred='A' ttft_ms=403.94
[14/800] ar

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# NEXT — SmolLM2 prompt_id only (concise + 5shot). Two 800-item runs; control skips.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "prompt_id")

cwd: /content/apertus-eval-prep
sweep in-process model=HuggingFaceTB/SmolLM2-1.7B-Instruct factor=prompt_id
run  SmolLM2-1.7B-Instruct_prompt_id_concise_a0852ca6fc3e5c08 factor=prompt_id=concise


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[1/800] arc_easy/eval/Mercury_SC_409038 correct=True pred='A' ttft_ms=267.61
[2/800] arc_easy/eval/MCAS_2001_5_19 correct=False pred='A' ttft_ms=175.34
[3/800] arc_easy/eval/NYSEDREGENTS_2008_4_21 correct=False pred='A' ttft_ms=178.05
[4/800] arc_easy/eval/Mercury_7115098 correct=True pred='A' ttft_ms=180.81
[5/800] arc_easy/eval/MCAS_2000_8_34 correct=False pred='B' ttft_ms=183.68
[6/800] arc_easy/eval/Mercury_SC_405883 correct=False pred='A' ttft_ms=179.94
[7/800] arc_easy/eval/Mercury_401762 correct=True pred='D' ttft_ms=188.06
[8/800] arc_easy/eval/Mercury_SC_410624 correct=True pred='A' ttft_ms=189.74
[9/800] arc_easy/eval/Mercury_SC_406688 correct=False pred='A' ttft_ms=186.11
[10/800] arc_easy/eval/Mercury_400556 correct=True pred='D' ttft_ms=187.93
[11/800] arc_easy/eval/Mercury_7219713 correct=True pred='D' ttft_ms=259.01
[12/800] arc_easy/eval/ACTAAP_2014_5_13 correct=True pred='A' ttft_ms=187.53
[13/800] arc_easy/eval/MCAS_2004_5_14 correct=True pred='A' ttft_ms=186.76
[14/8

In [ ]:
# Later — remaining SmolLM2 factors (control is skipped)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct")

In [ ]:
# Later — remaining Qwen 3B factors
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct")

In [ ]:
# Later — remaining Phi-3.5 factors
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct")

In [ ]:
# T4 only keeps 7B int4 (fp16 / int8 / vLLM are skipped)
sweep("--only-model", "Qwen/Qwen2.5-7B-Instruct")

## Report (only after several cells exist)

cwd must be `/content/apertus-eval-prep`. Do not run this instead of a sweep.

In [ ]:
from google.colab import files
from pathlib import Path

assert Path("results/registry_paper.jsonl").exists(), "No paper registry in this runtime. Restore from Drive first."
!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack the zip (or Drive folder) into the Mac clone. Commit `results/registry_paper.jsonl` and new `results/runs/*.json`. Do not edit numbers.